# LoRA Adapter from Scratch

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

LoRA freezes the pretrained weights $W \in \mathbb{R}^{d \times d}$ and learns a low-rank update $\Delta W = BA$ with $B \in \mathbb{R}^{d \times r}, A \in \mathbb{R}^{r \times d}$ and $r \ll d$. You train $O(rd)$ parameters instead of $O(d^2)$, and you can stash $A, B$ as a tiny adapter per task.


## Mathematical Formulation

$$h = Wx + \Delta W\,x = Wx + B(Ax),\quad \text{rank}(\Delta W) \leq r$$

We typically initialise $B = 0$ so the adapter starts as a no-op, and scale by $\alpha / r$ to keep updates well-conditioned.


## Implementation


In [ ]:
import torch
import torch.nn as nn


In [ ]:
class LoRALinear(nn.Module):
    """Wraps a frozen `nn.Linear` and adds a trainable low-rank update."""
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        in_f, out_f = base.in_features, base.out_features
        self.A = nn.Parameter(torch.randn(r, in_f) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_f, r))
        self.scale = alpha / r
    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A.T) @ self.B.T

def trainable_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


## Experiment


In [ ]:
# Wrap a 1024×1024 linear; compare param counts
base = nn.Linear(1024, 1024)
print('full Linear params:', sum(p.numel() for p in base.parameters()))
lora = LoRALinear(nn.Linear(1024, 1024), r=8)
print('LoRA trainable params:', trainable_params(lora))


In [ ]:
# Forward should match base initially (B = 0)
x = torch.randn(2, 1024)
assert torch.allclose(lora(x), lora.base(x))
print('LoRA initial output matches frozen base.')


## Discussion

- Init $B = 0$ guarantees the adapter is a no-op at start — equivalent to the base model.
- $r$ trades capacity for cost: 4–16 is common; very small tasks can use $r = 2$.
- You can merge $A,B$ back into $W$ at inference for zero-overhead deployment.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
